# Experiment: Bank Marketing Analysis

Objective:
- Build a simple end-to-end classification workflow for the bank marketing dataset.
- Use a `RandomForestClassifier` with sklearn preprocessing.
- Evaluate the model with F1 as the primary metric.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
DATA_PATH = Path("bank-full.csv")

plt.style.use("ggplot")
pd.set_option("display.max_columns", None)


## Plan

- Load the dataset and confirm the expected shape.
- Inspect class balance and a few important feature distributions.
- Exclude `duration` to avoid target leakage.
- One-hot encode categorical variables and train a balanced random forest.
- Report accuracy, precision, recall, F1, and a confusion matrix.


In [ ]:
df = pd.read_csv(DATA_PATH, sep=";")

print(f"Dataset shape: {df.shape}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")
print("\nTarget distribution:")
print(df["y"].value_counts())
print("\nTarget ratio:")
print(df["y"].value_counts(normalize=True).round(4))

df.head()


## Light EDA

The next cells provide a small visual scan of the target and a few core features.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df["y"].value_counts().plot(kind="bar", ax=axes[0, 0], color=["#4C72B0", "#55A868"])
axes[0, 0].set_title("Target Distribution")
axes[0, 0].set_xlabel("Subscribed")
axes[0, 0].set_ylabel("Count")

df["job"].value_counts().plot(kind="bar", ax=axes[0, 1], color="#4C72B0")
axes[0, 1].set_title("Job Distribution")
axes[0, 1].set_xlabel("Job")
axes[0, 1].set_ylabel("Count")
axes[0, 1].tick_params(axis="x", rotation=45)

df["education"].value_counts().plot(kind="bar", ax=axes[1, 0], color="#C44E52")
axes[1, 0].set_title("Education Distribution")
axes[1, 0].set_xlabel("Education")
axes[1, 0].set_ylabel("Count")
axes[1, 0].tick_params(axis="x", rotation=0)

df["marital"].value_counts().plot(kind="bar", ax=axes[1, 1], color="#8172B2")
axes[1, 1].set_title("Marital Status Distribution")
axes[1, 1].set_xlabel("Marital Status")
axes[1, 1].set_ylabel("Count")
axes[1, 1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df["age"].plot(kind="hist", bins=20, ax=axes[0], color="#4C72B0", edgecolor="black")
axes[0].set_title("Age Distribution")
axes[0].set_xlabel("Age")

df["balance"].plot(kind="hist", bins=30, ax=axes[1], color="#55A868", edgecolor="black")
axes[1].set_title("Balance Distribution")
axes[1].set_xlabel("Balance")

plt.tight_layout()
plt.show()


## Feature Setup And Train/Test Split

The `duration` column is removed because it is only known after a call ends, so keeping it would leak target information into the model.


In [ ]:
X = df.drop(columns=["y", "duration"]).copy()
y = df["y"].copy()

categorical_features = X.select_dtypes(include="object").columns.tolist()
numerical_features = X.select_dtypes(exclude="object").columns.tolist()

print("Duration excluded:", "duration" not in X.columns)
print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

train_distribution = y_train.value_counts(normalize=True).round(4)
test_distribution = y_test.value_counts(normalize=True).round(4)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain target ratio:")
print(train_distribution)
print("\nTest target ratio:")
print(test_distribution)


## Preprocessing And Model Training

Categorical columns are one-hot encoded. Numeric columns pass through unchanged.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numerical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                random_state=RANDOM_STATE,
                class_weight="balanced",
                n_jobs=1,
            ),
        ),
    ]
)

model.fit(X_train, y_train)
print("Model training completed.")


## Evaluation

F1 is treated as the main metric because the target classes are imbalanced.


In [ ]:
y_pred = model.predict(X_test)

metrics_summary = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, pos_label="yes"),
    "recall": recall_score(y_test, y_pred, pos_label="yes"),
    "f1": f1_score(y_test, y_pred, pos_label="yes"),
}

for metric_name, metric_value in metrics_summary.items():
    print(f"{metric_name.title()}: {metric_value:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=["no", "yes"])
print("Confusion Matrix:")
print(cm)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=["no", "yes"],
    cmap="Blues",
    ax=ax,
)
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()


## Results Summary

- The notebook loads the data, runs light EDA, trains one model, and reports standard classification metrics.
- If you want to improve recall or F1 further, the next step is to compare this model against logistic regression or gradient-boosted trees.
